# Save and reopen a run

<a id="save-and-reuse"></a>

An improvement run contains more than its selected source: it also has earlier
versions, measurements, and work counts. Save that history to disk so you can
close your notebook, reopen the result, and decide what to reuse.
Reopening reads recorded facts; resuming performs compatible unfinished work.
[Runs and history](https://sentient-xyz.github.io/meta-evolve-docs/concepts/history/) explains that distinction.

Here, we'll also export the selected parser and use it after the history is
closed. The next guide applies the same save/reopen calls to a search comparison.

| You want to… | Use |
|---|---|
| [Save a run as it executes](#save-the-run) | `storage=` with `Storage.durable()` |
| [Read it after restarting](#reopen-after-a-kernel-restart) | `open_run()` |
| [Reuse the selected source](#use-the-reviewed-source) | Export `best().value` |

The revisions are handwritten; Python executes and grades them. No model,
SDK, or credentials are needed.



<a id="set-up-this-page"></a>
<a id="complete-source"></a>
<a id="1-install"></a>
<a id="2-define-a-small-improvement-experiment"></a>

## Required setup for a fresh notebook

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

**Starting source and instructions.** `SEED` reads a duration's number but
ignores its unit. `CONTRACT` tells the proposer what the parser should do.

In [ ]:
import meta_evolve as meta

CONTRACT = """Implement parse_seconds(text) for whole-number durations.
Inputs contain a number followed by s, m, or h, such as '30s' or '2h'.
Return the duration in seconds as an integer. Return only Python source.
"""

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''

**Evaluation.** `evaluate` returns the fraction of six checks answered
correctly. `load_parser` executes the source locally; use this evaluator
with the reviewed, handwritten code shown here.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]


def evaluate(source):
    parse = load_parser(source)
    passed = 0
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is int and actual == expected:
            passed += 1
    return passed / len(CASES)


print(f"Starting score: {evaluate(SEED):.0%}")
# Output:
# Starting score: 33%

**Revisions.** The simulated agent adds minutes support, then hours support.

In [ ]:
MINUTES = '''def parse_seconds(text):
    quantity = int(text[:-1])
    return quantity * 60 if text[-1] == "m" else quantity
'''

COMPLETE = '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

**Proposer.** `propose` supplies the instructions and current source to the
simulated agent. Its response depends on the source, so reruns repeat the
same revisions without a hidden response counter.

In [ ]:
def agent(message):
    """Simulate source revisions, without a model or a call counter."""
    source = message.rsplit("Current source:\n", 1)[1]
    if source == SEED:
        return MINUTES
    if source in (MINUTES, COMPLETE):
        return COMPLETE
    raise ValueError("This simulation only recognizes the tutorial sources.")


def propose(source):
    message = f"{CONTRACT}\nCurrent source:\n{source}"
    return agent(message)

The [Start here walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/start-here/) introduces the starting parser.


<a id="save-the-run"></a>
<a id="save-a-seed-run-then-exit"></a>
<a id="save-a-model-produced-result"></a>

<a id="3-run-with-storage-on-disk"></a>

## 1. Run with storage on disk

[`meta.improve()`][meta_evolve.improve] evaluates the seed, proposes revisions
from the best version so far, and evaluates each returned source. Higher scalar
scores win by default. `trials=2` allows two proposals after the seed evaluation.

By default, the run's history lives in memory. [`meta.Storage.durable()`][meta_evolve.Storage.durable]
opens storage on disk. Passing it as `storage=` records the run there as it
executes. The `with` block keeps storage open for the run and closes it on exit.

Each execution below creates a new folder under `runs/`. The random suffix
keeps earlier runs separate; it does not affect the proposer's choices.

In [ ]:
from pathlib import Path
from uuid import uuid4

run_directory = Path("runs") / f"duration-{uuid4().hex[:12]}"
run_directory.mkdir(parents=True)

with meta.Storage.durable(run_directory / "history") as storage:
    saved = meta.improve(
        seed=SEED,
        proposer=propose,
        evaluator=evaluate,
        trials=2,
        storage=storage,
    )
    saved_usage = saved.usage()
    saved_snapshot = saved.summary()

Path("last-duration-run.txt").write_text(str(run_directory), encoding="utf-8")
print("Saved proposal attempts:", saved_usage.trials)
print("Saved evaluations:", saved_usage.evaluations)
saved_snapshot
# Output:
# Saved proposal attempts: 2
# Saved evaluations: 3

The final expression displays the captured report in a notebook, after the
storage block has closed. Use `print(saved_snapshot)` for plain text. Its
location describes where history was saved at capture; it does not check that
those files still exist. This local callable run is saved but does not declare
resume support. A fresh `saved.summary()` is a storage query and requires an
open handle; the next section shows how to reopen history.

`usage().trials` counts two proposal attempts after the seed;
`usage().evaluations` counts three evaluations, including the seed.
The earlier manual `evaluate(SEED)` preview is outside these counts.

`last-duration-run.txt` is a small note containing the latest run folder's path.
It lets the next cell find that folder after Python variables disappear.
Rerunning the saving cell updates the note and keeps earlier run folders.
The history holds the recorded artifacts and outcomes, not your notebook's
Python variables. Keep these files on persistent disk if your notebook service
discards local files when its runtime is deleted.

<a id="reopen-in-a-fresh-process"></a>
<a id="reopen-after-a-kernel-restart"></a>

<a id="4-restart-the-kernel-and-reopen"></a>

## 2. Restart the kernel and reopen

**Restart your notebook kernel now**, then run the next cell by itself in the
same working directory. Meta-Evolve must still be installed. This cell includes
all its imports and needs no `SEED`, proposer, or evaluator definitions.

[`meta.open_run()`][meta_evolve.open_run] opens the only run in this folder for
inspection, without proposing or evaluating source. It validates the history
before entering the block and closes storage on exit, including on error.
An empty store raises `RunNotFound`; multiple runs raise `RunSelectionRequired`
with available IDs. Pass `run_id=...` to choose explicitly: content-derived ID
order does not tell you which run is newest.

`trials()` lists the seed and attempted revisions. `best_trial()` returns the selected
attempt and its measurements; `best().value` gives the selected source string.
Read these while the storage handle is open:

In [ ]:
from pathlib import Path
import meta_evolve as meta

run_directory = Path(Path("last-duration-run.txt").read_text(encoding="utf-8"))
with meta.open_run(run_directory / "history") as reopened:
    for number, attempt in enumerate(reopened.trials()):
        print(f"Version {number}: {attempt.metrics['score']:.0%}")
    selected_source = reopened.best().value
    print(f"Retained score: {reopened.best_trial().metrics['score']:.0%}")
    print("Retained evaluations:", reopened.usage().evaluations)
# Output:
# Version 0: 33%
# Version 1: 67%
# Version 2: 100%
# Retained score: 100%
# Retained evaluations: 3

The same three measurements are still available after the original process exits.
Reopening reads recorded results; it does not spend more evaluations or rerun a
model. The selected source is now an ordinary string you can use after storage
closes. To inspect again, reopen the storage handle as above.

If `last-duration-run.txt` is missing, check that section 1 completed and that
you are in the same working directory. To open an older run, replace the line
that reads the note with
`run_directory = Path("runs/duration-YOUR-SAVED-SUFFIX")`, using its actual folder.
Its original run ID is still in `storage.runs()`; reopening does not create one.

Saving local functions does not make the run resumable. **Reopen** means reading
past results; **resume** means continuing unfinished work and requires
reconstructible components. See the [durability reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/durability/#resume)
when you need that separate capability.

<a id="reuse-the-reviewed-source"></a>
<a id="use-the-reviewed-source"></a>

<a id="5-export-and-use-the-selected-source"></a>

## 3. Export and use the selected source

Write the retained source to a normal Python file, then print it for review.
Neither writing nor printing source executes it.

In [ ]:
export_path = run_directory / "selected.py"
export_path.write_text(selected_source, encoding="utf-8")
print(export_path.read_text(encoding="utf-8"), end="")
# Output:
# def parse_seconds(text):
#     quantity = int(text[:-1])
#     seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
#     return quantity * seconds_per_unit[text[-1]]

The source is the handwritten `COMPLETE` revision from the required setup.
Load that reviewed file and call its parser. **This cell executes the exported code.**
It needs no live run or open storage handle.

In [ ]:
namespace = {}
exec(export_path.read_text(encoding="utf-8"), namespace)
parse_seconds = namespace["parse_seconds"]
print("4m:", parse_seconds("4m"))
print("2h:", parse_seconds("2h"))
# Output:
# 4m: 240
# 2h: 7200

Keep these pieces according to what you need:

| File or folder | What it preserves |
|---|---|
| `runs/duration-…/history/` | The full recorded run: source versions, measurements, lineage, work counts, and any evidence or failures |
| `runs/duration-…/selected.py` | Just the selected source, ready to copy into another project |
| `last-duration-run.txt` | The path used to find the most recently saved run folder |

An export does not contain the evaluation history. Keep the whole `history/`
directory when you want to reopen the run. You can regenerate `selected.py`
from that history by rerunning sections 2 and 3.

## Change and predict

After the restart, rerun the setup definition cells
(the installation need not repeat). Keep a note of the earlier `run_directory`.
Change `trials=2` to `trials=1` in section 1 and predict the retained score and
work count. Save another run, restart again, then reopen and export using
sections 2 and 3.

The reopened history now has **two versions**, scoring 33% and 67%, and **two
evaluations**. The exported parser still returns `240` for `"4m"`, but returns
`2` for `"2h"`: one proposal added minutes support only. The earlier 100% run
remains in its own folder. Reopen that earlier folder using its saved path:
its three versions and 100% result are unchanged. A successful example input
is not a substitute for the full set of checks.

## Do I need my original functions to reopen?

No. The reopening cell reads recorded artifacts, measurements, and usage;
it needs only the installed package and saved history. It makes no model
requests. Continuing unfinished work is a separate operation: local notebook
functions are not automatically reconstructible for resume.

For your own experiment, pass durable storage to `improve()` or `run()` and keep
both its location and run ID. After reopening, use the selected value in the
way its type requires; this example exports Python because its artifact is source.
Custom artifact values additionally need their codec registry via `registry=`.

## Choose a run in a shared store

This optional example saves two small experiments in a separate shared store.
Keep the first run's ID as well as its directory so that reopening selects the
intended experiment. The parser history and its export remain in their folders.

In [ ]:
from pathlib import Path
import meta_evolve as meta

run_directory = Path(Path("last-duration-run.txt").read_text(encoding="utf-8"))
with meta.Storage.durable(run_directory / "shared-history") as storage:
    chosen = meta.improve(
        seed=0, proposer=lambda value: value + 1, evaluator=float,
        trials=1, storage=storage,
    )
    meta.improve(
        seed=10, proposer=lambda value: value + 1, evaluator=float,
        trials=1, storage=storage,
    )
(run_directory / "chosen-run.txt").write_text(chosen.id.value, encoding="utf-8")

Restart the kernel again, then run this cell. It reads the ID from the file;
neither list position nor score decides which experiment to open.

In [ ]:
from pathlib import Path
import meta_evolve as meta

run_directory = Path(Path("last-duration-run.txt").read_text(encoding="utf-8"))
chosen_id = (run_directory / "chosen-run.txt").read_text(encoding="utf-8")
with meta.open_run(run_directory / "shared-history", run_id=chosen_id) as reopened:
    chosen_value = reopened.best().value
print("Explicitly selected:", chosen_value)
# Output:
# Explicitly selected: 1

Next, [choose what to try next](https://sentient-xyz.github.io/meta-evolve-docs/learn/02-change-search/). With evaluation, feedback,
and saved history in place, use a small branching example to see when another
search strategy helps. The
[durability reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/durability/) covers recovery and resume.